# PySpark MySQL Connectivity

This notebook demonstrates how to connect PySpark with MySQL database for reading and writing data.

In [ ]:
# Install and setup Java (for Google Colab)
import os

def install_java():
    !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    !java -version

install_java()

In [ ]:
# Install PySpark
!pip install pyspark

## Download MySQL JDBC Driver

In [ ]:
# Download MySQL JDBC connector
!wget https://repo1.maven.org/maven2/mysql/mysql-connector-java/8.0.28/mysql-connector-java-8.0.28.jar -O /content/mysql-connector-java-8.0.28.jar

# Verify download
!ls -lh /content/mysql-connector-java-8.0.28.jar

## Create Spark Session with MySQL Driver

In [ ]:
from pyspark.sql import SparkSession

# Create Spark session with MySQL JDBC driver
spark = SparkSession.builder \
    .appName('MySQL PySpark Connector') \
    .config("spark.jars", "/content/mysql-connector-java-8.0.28.jar") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

## Database Connection Properties

In [ ]:
# MySQL connection properties
# IMPORTANT: Update these with your actual database credentials

mysql_host = "localhost"          # or IP address
mysql_port = "3306"               # default MySQL port
mysql_database = "cdac_database"  # your database name
mysql_user = "root"               # your username
mysql_password = "your_password"  # your password

# JDBC URL
jdbc_url = f"jdbc:mysql://{mysql_host}:{mysql_port}/{mysql_database}"

# Connection properties
connection_properties = {
    "user": mysql_user,
    "password": mysql_password,
    "driver": "com.mysql.cj.jdbc.Driver"
}

print(f"JDBC URL: {jdbc_url}")
print(f"Driver: {connection_properties['driver']}")

## Reading Data from MySQL

### Method 1: Read Entire Table

In [ ]:
# Read entire table from MySQL
table_name = "employees"  # Replace with your table name

df_employees = spark.read \
    .jdbc(
        url=jdbc_url,
        table=table_name,
        properties=connection_properties
    )

print(f"Table: {table_name}")
print(f"Record count: {df_employees.count()}")
print("\nSchema:")
df_employees.printSchema()

print("\nSample data:")
df_employees.show(5)

### Method 2: Read with SQL Query

In [ ]:
# Read using custom SQL query
sql_query = """
    (SELECT 
        employee_id, 
        first_name, 
        last_name, 
        salary,
        department
    FROM employees 
    WHERE salary > 50000) AS high_earners
"""

df_high_earners = spark.read \
    .jdbc(
        url=jdbc_url,
        table=sql_query,
        properties=connection_properties
    )

print("High earners (salary > 50000):")
df_high_earners.show(10)

### Method 3: Read with Partitioning for Performance

In [ ]:
# Read large table with parallel partitions
# This improves performance by reading in parallel

df_partitioned = spark.read \
    .jdbc(
        url=jdbc_url,
        table="employees",
        column="employee_id",      # Column to partition on (must be numeric)
        lowerBound=1,               # Minimum value
        upperBound=10000,           # Maximum value
        numPartitions=4,            # Number of parallel connections
        properties=connection_properties
    )

print(f"Number of partitions: {df_partitioned.rdd.getNumPartitions()}")
df_partitioned.show(5)

## Transform Data with PySpark

In [ ]:
from pyspark.sql.functions import col, avg, sum as spark_sum, count, round as spark_round

# Register as temp view for SQL queries
df_employees.createOrReplaceTempView("employees")

# Example transformations

# 1. Department-wise statistics
dept_stats = spark.sql("""
    SELECT 
        department,
        COUNT(*) as employee_count,
        ROUND(AVG(salary), 2) as avg_salary,
        SUM(salary) as total_salary
    FROM employees
    GROUP BY department
    ORDER BY total_salary DESC
""")

print("Department-wise Statistics:")
dept_stats.show()

In [ ]:
# 2. Add bonus column (10% of salary)
df_with_bonus = df_employees \
    .withColumn("bonus", col("salary") * 0.10) \
    .withColumn("total_compensation", col("salary") + col("bonus"))

print("Employees with Bonus:")
df_with_bonus.select(
    "employee_id", "first_name", "last_name", "salary", "bonus", "total_compensation"
).show(10)

## Writing Data to MySQL

### Method 1: Overwrite Existing Table

In [ ]:
# Write DataFrame to MySQL (overwrite mode)
# WARNING: This will replace existing table!

dept_stats.write \
    .jdbc(
        url=jdbc_url,
        table="department_statistics",
        mode="overwrite",              # Options: overwrite, append, ignore, error
        properties=connection_properties
    )

print("Department statistics written to MySQL table: department_statistics")

### Method 2: Append to Existing Table

In [ ]:
# Append data to existing table
df_with_bonus.write \
    .jdbc(
        url=jdbc_url,
        table="employee_compensation",
        mode="append",                 # Adds rows without deleting existing data
        properties=connection_properties
    )

print("Employee compensation data appended to MySQL table: employee_compensation")

### Method 3: Write with Custom Properties

In [ ]:
# Write with custom batch size and isolation level
custom_props = connection_properties.copy()
custom_props["batchsize"] = "10000"           # Rows per batch
custom_props["isolationLevel"] = "READ_COMMITTED"

df_high_earners.write \
    .jdbc(
        url=jdbc_url,
        table="high_earners",
        mode="overwrite",
        properties=custom_props
    )

print("High earners data written with custom properties")

## Complete ETL Example

In [ ]:
# Complete ETL pipeline: Extract, Transform, Load

print("Starting ETL Pipeline...")

# EXTRACT: Read from MySQL
print("\n1. EXTRACT: Reading from MySQL...")
source_df = spark.read \
    .jdbc(
        url=jdbc_url,
        table="employees",
        properties=connection_properties
    )
print(f"   Extracted {source_df.count()} records")

# TRANSFORM: Process data
print("\n2. TRANSFORM: Processing data...")
transformed_df = source_df \
    .filter(col("salary") > 30000) \
    .withColumn("salary_grade", 
        F.when(col("salary") > 100000, "A")
         .when(col("salary") > 70000, "B")
         .when(col("salary") > 50000, "C")
         .otherwise("D")
    ) \
    .withColumn("annual_bonus", col("salary") * 0.15)

print(f"   Transformed {transformed_df.count()} records")
transformed_df.show(5)

# LOAD: Write back to MySQL
print("\n3. LOAD: Writing to MySQL...")
transformed_df.write \
    .jdbc(
        url=jdbc_url,
        table="employee_analysis",
        mode="overwrite",
        properties=connection_properties
    )
print("   Data written to employee_analysis table")

print("\nETL Pipeline completed successfully!")

## Execute Custom SQL on MySQL

In [ ]:
# Execute complex SQL query
complex_query = """
    (SELECT 
        e.department,
        e.first_name,
        e.last_name,
        e.salary,
        d.dept_name,
        d.location
    FROM employees e
    JOIN departments d ON e.department = d.dept_id
    WHERE e.salary > 60000
    ORDER BY e.salary DESC) AS result
"""

df_complex = spark.read \
    .jdbc(
        url=jdbc_url,
        table=complex_query,
        properties=connection_properties
    )

print("Complex Query Results:")
df_complex.show(10, truncate=False)

## Error Handling

In [ ]:
# Robust connection with error handling
def read_mysql_table_safe(table_name):
    """Safely read MySQL table with error handling"""
    try:
        df = spark.read \
            .jdbc(
                url=jdbc_url,
                table=table_name,
                properties=connection_properties
            )
        print(f"Successfully read table: {table_name}")
        return df
    except Exception as e:
        print(f"Error reading table {table_name}: {str(e)}")
        return None

# Test the function
df = read_mysql_table_safe("employees")
if df:
    df.show(5)

## List All Tables in Database

In [ ]:
# Query to list all tables
tables_query = "(SHOW TABLES) AS tables"

try:
    df_tables = spark.read \
        .jdbc(
            url=jdbc_url,
            table=tables_query,
            properties=connection_properties
        )
    
    print(f"Tables in database '{mysql_database}':")
    df_tables.show(truncate=False)
except Exception as e:
    print(f"Error listing tables: {str(e)}")

## Key Points to Remember

### Connection:
1. Always include MySQL JDBC driver in Spark config
2. Use proper JDBC URL format: `jdbc:mysql://host:port/database`
3. Include driver class: `com.mysql.cj.jdbc.Driver`

### Performance:
1. Use partitioning for large tables (column, lowerBound, upperBound, numPartitions)
2. Choose numeric columns for partitioning
3. Set appropriate batch size for writes

### Write Modes:
- **overwrite**: Replaces table completely
- **append**: Adds rows to existing table
- **ignore**: Writes only if table doesn't exist
- **error**: Throws error if table exists (default)

### Security:
1. Never hardcode credentials in production
2. Use environment variables or secrets management
3. Enable SSL for production connections

In [ ]:
# Stop Spark Session
spark.stop()